# Solutions · 8. Partial least squares for soft sensors

Worked solutions to the three exercises of [Lecture 8](../../notebooks/part3_chemometrics/08_partial_least_squares_nipals_for_soft_sensors.ipynb). Each solution is computed here and checked with `assert` statements against an exact identity or an independent calculation. The first code cell sets up the notebook and repeats the demonstrations of the lecture without printing them, because the solutions use their variables.

In [1]:
try:                      # use the installed package, or the copy in this repository
    import engml_2_applied
except ImportError:
    import pathlib, subprocess, sys
    for _root in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_root / "engml_2_applied" / "__init__.py").exists():
            sys.path.insert(0, str(_root))
            break
    else:                 # on Google Colab: install from GitHub
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "git+https://github.com/ktwyw/engml-2-applied.git"], check=True)
    import engml_2_applied

import inspect
import numpy as np
import matplotlib.pyplot as plt
from engml_2_applied import (clustering_phases, constants, descriptors, dim_reduction, linear_physics,
                             trees_ensembles, uncertainty)
from engml_2_applied.plotting import panel_label, use_style

use_style()

def show_source(obj):
    """Return the source code of a library function or class."""
    return inspect.getsource(obj)

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

import io, sys
_stdout, sys.stdout = sys.stdout, io.StringIO()   # the demonstrations are repeated without their output
heading("Demonstration 1: Wold NIPALS PLS1 on a 120-Wavelength NIR Soft Sensor (d = 120 >> N = 35)")
rng = np.random.default_rng(42)
N, d = 35, 120
wl = np.linspace(1100.0, 2300.0, d)
# 3 overlapping chemical chromophores (water, methanol, ethanol):
E_pure = np.vstack([
    np.exp(-0.5 * ((wl - 1450.0) / 60.0) ** 2) + 0.4 * np.exp(-0.5 * ((wl - 1940.0) / 50.0) ** 2),
    np.exp(-0.5 * ((wl - 1680.0) / 55.0) ** 2) + 0.6 * np.exp(-0.5 * ((wl - 2100.0) / 45.0) ** 2),
    np.exp(-0.5 * ((wl - 1720.0) / 50.0) ** 2) + 0.5 * np.exp(-0.5 * ((wl - 2150.0) / 55.0) ** 2),
])
C_conc = rng.uniform(0.05, 0.95, size=(N, 3))
X_nir = C_conc @ E_pure + rng.normal(0.0, 0.008, size=(N, d))
y_meoh = C_conc[:, 1] + rng.normal(0.0, 0.06, size=N)

pls3 = dim_reduction.pls1_nipals(X_nir, y_meoh, n_components=3)
T_scores = pls3["scores"]
G_scores = T_scores.T @ T_scores
off_diag = G_scores - np.diag(np.diag(G_scores))
print("Score inner-product matrix T^T T:\n", np.round(G_scores, 4))
print("Max off-diagonal score inner product:", float(np.max(np.abs(off_diag))))
assert np.max(np.abs(off_diag)) < 1e-11
heading("Demonstration 2: Step-by-Step NIPALS Deflation Energy & P^T W Structure")
E_curr = X_nir - np.mean(X_nir, axis=0)
f_curr = y_meoh - float(np.mean(y_meoh))
e0_norm = float(np.sum(E_curr ** 2))
f0_norm = float(np.sum(f_curr ** 2))
W_m, P_m = np.zeros((d, 3)), np.zeros((d, 3))
for a in range(3):
    w_a = E_curr.T @ f_curr
    w_a /= np.linalg.norm(w_a)
    t_a = E_curr @ w_a
    p_a = (E_curr.T @ t_a) / float(t_a @ t_a)
    q_a = float(f_curr @ t_a) / float(t_a @ t_a)
    E_curr = E_curr - np.outer(t_a, p_a)
    f_curr = f_curr - q_a * t_a
    W_m[:, a], P_m[:, a] = w_a, p_a
    print(f"After LV {a+1} | Spectral X variance captured = {100*(1 - np.sum(E_curr**2)/e0_norm):6.2f}% | Target y variance captured = {100*(1 - np.sum(f_curr**2)/f0_norm):6.2f}%")
print("\nUpper-triangular matrix P^T W:\n", np.round(P_m.T @ W_m, 6))
assert np.allclose(np.tril(P_m.T @ W_m, -1), 0.0, atol=1e-12) and np.allclose(np.diag(P_m.T @ W_m), 1.0, atol=1e-12)
heading("Demonstration 3: Out-of-Sample NIR Soft-Sensor Benchmark (PLS1 vs. Minimum-Norm OLS)")
C_test = rng.uniform(0.05, 0.95, size=(40, 3))
X_test = C_test @ E_pure + rng.normal(0.0, 0.008, size=(40, d))
y_test = C_test[:, 1]

y_pred_pls = X_test @ pls3["coef"] + pls3["intercept"]
X_tr_aug = np.column_stack([np.ones(N), X_nir])
X_te_aug = np.column_stack([np.ones(40), X_test])
w_pinv = np.linalg.lstsq(X_tr_aug, y_meoh, rcond=None)[0]
y_pred_ols = X_te_aug @ w_pinv

rmse_pls = float(np.sqrt(np.mean((y_pred_pls - y_test) ** 2)))
rmse_ols = float(np.sqrt(np.mean((y_pred_ols - y_test) ** 2)))
print(f"Training Set Fit RMSE (N=35)   : Minimum-norm OLS = {np.sqrt(np.mean((X_tr_aug @ w_pinv - y_meoh)**2)):.2e} (Interpolates noise) | PLS1 = {np.sqrt(np.mean((pls3['y_pred'] - y_meoh)**2)):.4f}")
print(f"Out-of-Sample Test RMSE (N=40) : Minimum-norm OLS = {rmse_ols:.4f} -> 3-Component PLS1 = {rmse_pls:.4f}")
assert rmse_pls < 0.08 and rmse_pls < 0.65 * rmse_ols
heading("Demonstration 4: Latent Variable Count k Sweep (Underfitting -> Optimal Rank 3 -> Overfitting)")
print(f"{'Latent Vars k':>13s} | {'Train RMSE':>12s} | {'Test RMSE':>12s} | {'||beta_PLS||_2':>14s} | {'Regime':>22s}")
print("-" * 82)
for k_lv in [1, 2, 3, 4, 6, 10]:
    fit_k = dim_reduction.pls1_nipals(X_nir, y_meoh, n_components=k_lv)
    tr_err = float(np.sqrt(np.mean((fit_k["y_pred"] - y_meoh) ** 2)))
    pred_k = X_test @ fit_k["coef"] + fit_k["intercept"]
    te_err = float(np.sqrt(np.mean((pred_k - y_test) ** 2)))
    b_norm = float(np.linalg.norm(fit_k["coef"]))
    regime = "Underfit (< 3 species)" if k_lv < 3 else ("Optimal (k* = 3)" if k_lv == 3 else "Overfitting GC noise")
    print(f"{k_lv:13d} | {tr_err:12.5f} | {te_err:12.5f} | {b_norm:14.4f} | {regime:>22s}")
heading("Inside the Algorithm: By-Hand W(P^T W)^(-1)q vs. the Library")
q_hand = np.linalg.solve(T_scores.T @ T_scores, T_scores.T @ (y_meoh - np.mean(y_meoh)))
beta_pls_hand = W_m @ np.linalg.solve(P_m.T @ W_m, q_hand)
diff_pls = float(np.max(np.abs(beta_pls_hand - pls3["coef"])))
print("by hand (pure NumPy) beta[:5]:", np.round(beta_pls_hand[:5], 6))
print("library (engml_2_applied)      beta[:5]:", np.round(pls3["coef"][:5], 6))
print(f"Max discrepancy              : {diff_pls:.2e}")
assert diff_pls < 1e-14
sys.stdout = _stdout
print("Set-up complete: the variables of the lecture's demonstrations are defined.")

Set-up complete: the variables of the lecture's demonstrations are defined.


## Exercise 1

**Problem.** Verify on a tall well-conditioned design matrix $\mathbf{X} \in \mathbb{R}^{40 \times 4}$ that `pls1_nipals(X, y, n_components=4)` (when $k = d$) converges to the exact ordinary least squares regression coefficients and intercept to within $10^{-11}$.

In [2]:
heading("Exercise 1: Exact Equivalence of Full-Rank PLS1 (k = d) and OLS")
X_tall = rng.normal(0.0, 1.0, size=(40, 4))
y_tall = X_tall @ np.array([1.5, -0.8, 2.2, -0.4]) + 3.0 + rng.normal(0.0, 0.1, size=40)
pls_d = dim_reduction.pls1_nipals(X_tall, y_tall, n_components=4)
w_ols_d = np.linalg.lstsq(np.column_stack([np.ones(40), X_tall]), y_tall, rcond=None)[0]

assert np.allclose(pls_d["coef"], w_ols_d[1:], atol=1e-11)
assert np.isclose(pls_d["intercept"], w_ols_d[0], atol=1e-11)
print("Verified exact equivalence of PLS1 (k = d = 4) and OLS:", np.round(pls_d["coef"], 5))


Exercise 1: Exact Equivalence of Full-Rank PLS1 (k = d) and OLS
Verified exact equivalence of PLS1 (k = d = 4) and OLS: [ 1.49605 -0.81297  2.21348 -0.40585]


**Discussion.** PLS1 searches the $k$-dimensional Krylov subspace $\mathcal{K}_k(\mathbf{X}_c^\top\mathbf{X}_c, \mathbf{X}_c^\top\mathbf{y}_c)$. When $k = d$, the Krylov subspace spans the entire column space of $\mathbf{X}_c^\top$, recovering the exact OLS estimator.

---

## Exercise 2

**Problem.** Make methanol a minor absorber: replace its pure spectrum (`E_pure[1]`) by a weak band, `0.08 * np.exp(-0.5 * ((wl - 1250.0) / 30.0) ** 2)`, rebuild the spectra, and compare PCR and PLS1 with 1, 2 and 3 components for the methanol concentration. Does PLS1 find the minor component with fewer components than PCR?

In [3]:
heading("Exercise 2: A minor absorber: PCR and PLS1 with 1, 2 and 3 components")
E_trace_meoh = E_pure.copy()
E_trace_meoh[1] = 0.08 * np.exp(-0.5 * ((wl - 1250.0) / 30.0) ** 2)
X_tr_minor = C_conc @ E_trace_meoh + rng.normal(0.0, 0.001, size=(N, d))
y_tr_minor = C_conc[:, 1]

rmse_minor = {}
for k in (1, 2, 3):
    pcr_m = dim_reduction.pcr_regression(X_tr_minor, y_tr_minor, n_components=k)
    pls_m = dim_reduction.pls1_nipals(X_tr_minor, y_tr_minor, n_components=k)
    rmse_minor[k] = (float(np.sqrt(np.mean((pcr_m["y_pred"] - y_tr_minor) ** 2))),
                     float(np.sqrt(np.mean((pls_m["y_pred"] - y_tr_minor) ** 2))))
    print(f"k = {k} | calibration RMSE: PCR = {rmse_minor[k][0]:.4f} | PLS1 = {rmse_minor[k][1]:.4f}")
print(f"standard deviation of the methanol concentration = {np.std(y_tr_minor):.4f}")
assert all(rmse_minor[k][1] <= rmse_minor[k][0] + 1e-12 for k in (1, 2, 3))
assert rmse_minor[3][0] < 0.02 and rmse_minor[2][1] > 0.1


Exercise 2: A minor absorber: PCR and PLS1 with 1, 2 and 3 components
k = 1 | calibration RMSE: PCR = 0.2560 | PLS1 = 0.2538
k = 2 | calibration RMSE: PCR = 0.2547 | PLS1 = 0.2360
k = 3 | calibration RMSE: PCR = 0.0042 | PLS1 = 0.0042
standard deviation of the methanol concentration = 0.2571


**Discussion.** The answer here is no. With one or two components both methods do little better than predicting the mean: PLS1 is slightly ahead with two components ($0.236$ against $0.255$), and neither is useful. With three components, one for each absorbing species, both reach $0.004$. PLS1 chooses directions by their covariance with $\mathbf{y}$, and covariance is correlation times the standard deviations: a direction of large spectral variance and modest correlation can still come before a weak band that is perfectly correlated. PLS1 never does worse than PCR with the same number of components, but the number of components must still be at least the number of independent sources of variation in the spectra, and should be chosen by cross-validation.

---

## Exercise 3

**Problem.** Implement **variable importance in projection (VIP)** scores `pls_vip_scores(X: np.ndarray, y: np.ndarray, n_components: int = 3) -> np.ndarray`, defined by $\mathrm{VIP}_j = \sqrt{d \sum_{a=1}^k (q_a^2 \mathbf{t}_a^\top\mathbf{t}_a)\,w_{j,a}^2 / \sum_{a=1}^k q_a^2 \mathbf{t}_a^\top\mathbf{t}_a}$, which satisfies the exact sum-of-squares identity $\frac{1}{d}\sum_{j=1}^d \mathrm{VIP}_j^2 = 1.0$. Verify on `(X_nir, y_meoh)` that $\frac{1}{d}\sum_j \mathrm{VIP}_j^2 = 1.0$ and that the peak VIP wavelength matches the true methanol absorption band ($1680\;\mathrm{nm}$ or $2100\;\mathrm{nm}$).

In [4]:
heading("Exercise 3: Implement PLS Variable Importance in Projection (VIP) From Scratch")
def pls_vip_scores(X: np.ndarray, y: np.ndarray, n_components: int = 3) -> np.ndarray:
    """Compute Wold's Variable Importance in Projection (VIP) wavelength scores for PLS1."""
    E_c = np.asarray(X, dtype=float) - np.mean(X, axis=0)
    f_c = np.asarray(y, dtype=float).ravel() - float(np.mean(y))
    n_s, d_s = E_c.shape
    W_vip = np.zeros((d_s, n_components))
    ss_y = np.zeros(n_components)
    for a in range(n_components):
        w_a = E_c.T @ f_c
        w_a /= np.linalg.norm(w_a)
        t_a = E_c @ w_a
        t_sq = float(t_a @ t_a)
        p_a = (E_c.T @ t_a) / t_sq
        q_a = float(f_c @ t_a) / t_sq
        E_c = E_c - np.outer(t_a, p_a)
        f_c = f_c - q_a * t_a
        W_vip[:, a] = w_a
        ss_y[a] = (q_a ** 2) * t_sq
    vip_sq = d_s * ((W_vip ** 2) @ ss_y) / max(float(np.sum(ss_y)), 1e-15)
    return np.sqrt(vip_sq)

vip = pls_vip_scores(X_nir, y_meoh, n_components=3)
peak_wl = float(wl[int(np.argmax(vip))])
print(f"Mean squared VIP score (1/d) sum(VIP_j^2) = {np.mean(vip ** 2):.10f} (Exact identity = 1.0)")
print(f"Top VIP wavelength = {peak_wl:.1f} nm (True methanol chromophore peaks = 1680 nm & 2100 nm)")
assert np.isclose(np.mean(vip ** 2), 1.0, rtol=1e-13) and (abs(peak_wl - 1680.0) < 35.0 or abs(peak_wl - 2100.0) < 35.0)


Exercise 3: Implement PLS Variable Importance in Projection (VIP) From Scratch
Mean squared VIP score (1/d) sum(VIP_j^2) = 1.0000000000 (Exact identity = 1.0)
Top VIP wavelength = 1664.7 nm (True methanol chromophore peaks = 1680 nm & 2100 nm)


**Discussion.** Wold's VIP scores ($\mathrm{VIP}_j > 1$) identify which physical wavelengths carry the strongest target covariance across all PLS latent variables while satisfying $\frac{1}{d}\sum_{j=1}^d \mathrm{VIP}_j^2 \equiv 1$.